# Implement Vector Search in Elastic using different embedding models

# 🔍 Vector Search

**Author:** Pradeep Pujari  
**Date:** August 18, 2025  
**Description:** This notebook Elasticsearch-based vector/semantic and hybrid search strategies on the product catalog index.

The workbook implements Vector Search in Elasticsearch using a simple dataset and
1. E5
2. Sentence BERT (Hugging Face) from Elastic Search Hub, because of:  
   a. Hugging Face has a dynamic community and I am worried the model changes may break the existing code  
   b. I believe Elastic Search Team take the stable model from Hugging Face and put into their HUB  
   c. Probably, we have support contract with Elastic Search (ES)

**Data:**  
_id  
name  
searchAttributes  
stringFacets  

## Prerequisities
Data collection. Collected about 100 records as below. 

## Performed a search as below in DEV index
<pre>
GET /catalog-read/_search
{
  "size": 100,
  "query": {
    "query_string": {
      "query": "\"rocky mountain\""
    }
  },
  "_source": ["_id", "name", "searchAttributes", "longdescription"]
}
</pre>
**STEPS**  
✅ Get data from Elastic Search

✅ Clean data like blank title etc.

✅ Reformat the data in "Item: {name}. Brand: {brand}. Color: {color}. Size: {size}. Category: {category}."

✅ Deploy Model to ES cluster (ONE TIME ONLY)

✅ Index mapping

✅ Ingest pipeline

✅ Vector generation with Sentence BERT and E5 (USE MODEL_ID)

✅ Indexing documents

✅ Hybrid search query

In [50]:
# import modules
import os
import re
import json
import torch
from getpass import getpass
from elasticsearch import Elasticsearch, helpers
import pandas as pd
import numpy as np
from elasticsearch.helpers import bulk, BulkIndexError
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

from pathlib import Path
from eland.ml.pytorch import PyTorchModel
from eland.ml.pytorch.transformers import TransformerModel

from dotenv import load_dotenv
load_dotenv()

True

In [51]:
# Load raw file as text
with open('../data/es_query_dataset.json', 'r', encoding='utf-8') as f:
    raw_data = f.read()

# Fix triple quotes ("""string""") to single quote format
raw_data = re.sub(r'"""\s*(.*?)\s*"""', lambda m: '"' + m.group(1).replace('"', '\\"') + '"', raw_data, flags=re.DOTALL)

# Optionally write back to a clean file
with open('../data/es_query_dataset_clean.json', 'w', encoding='utf-8') as f:
    f.write(raw_data)


In [52]:
# Load raw results from Elasticsearch
with open("../data/es_query_dataset_clean.json", 'r', encoding='utf-8') as f:
    data = json.load(f)

# Extract _id and cleaned _source (remove 'searchAttributes')
cleaned = []
for doc in data:
    source = {k: v for k, v in doc["_source"].items() if k != "searchAttributes"}
    source["_id"] = doc["_id"]  # Add the _id from Elasticsearch
    cleaned.append(source)

# Save cleaned data to a new JSON file (optional)
with open("../data/cleaned_results.json", "w") as out_file:
    json.dump(cleaned, out_file, indent=2)

# Load into DataFrame
df = pd.DataFrame(cleaned)

# Preview
df.head()


,stringFacets,name,_id
0,"[{'identifier': 'X_BRAND', 'value': 'Rocky Mou...",Rocky Mountain Tackle Signature Dodger,21RMAURCKYMTNDDGRFAC
1,"[{'identifier': 'X_BRAND', 'value': 'Rocky Mou...",Rocky Mountain Growler 20 Microshift Bike,23NFKAGRWLR20MCRSPRF
2,"[{'identifier': 'X_BRAND', 'value': 'Rocky Mou...",Rocky Mountain Instinct Powerplay Alloy 30,23NFKUNSTNCTPPLLYBAC
3,"[{'identifier': '320', 'storeId': 10001, 'part...",Rocky Mountain Tackle Signature Dodger,13502535
4,"[{'identifier': '3232', 'storeId': 10001, 'par...",Elakai Travel 1' x 2' Cornhole Boards with Car...,25353984


In [53]:
print(df[~df['stringFacets'].apply(lambda x: isinstance(x, list))])


    stringFacets                          name                _id
154          NaN  Nike Grind Dumbbell Rack Set  24NIKEGRDDBBUNDLE


In [54]:
df = df[df['stringFacets'].apply(lambda x: isinstance(x, list))].copy()

In [55]:

#mapping numeric identifiers to readable labels
identifier_map = {
    'X_BRAND': 'Brand',
    '5495': 'Gender',
    '5382': 'Subcategory',
    '4285': 'Category'
}

# Function to parse and flatten
def flatten_row(row):
    facets = row['stringFacets']
    info = {'Item': row['name']}
    info = {'_id' : row['_id']}
    
    for facet in facets:
        key = identifier_map.get(facet['identifier'])
        if key:
            info[key] = facet['value']
    
    # Build formatted string
    ordered_keys = ['Item', 'Brand', 'Gender', 'Subcategory', 'Category']
    return '. '.join(f"{k}: {info[k]}" for k in ordered_keys if k in info) + '.'

# Apply flattening
pd.set_option('display.max_colwidth', None)
df['flat_description'] = df.apply(flatten_row, axis=1)
print(df[['flat_description']])

pd.reset_option('display.max_colwidth')

                                                                                      flat_description
0    Brand: Rocky Mountain Tackle. Gender: Unisex. Subcategory: Dodgers & Flashers. Category: Fishing.
1                         Brand: Rocky Mountain. Gender: Men's. Subcategory: Bikes. Category: Cycling.
2                         Brand: Rocky Mountain. Gender: Men's. Subcategory: Bikes. Category: Cycling.
3    Brand: Rocky Mountain Tackle. Gender: Unisex. Subcategory: Dodgers & Flashers. Category: Fishing.
4                Brand: Elakai. Gender: Unisex. Subcategory: Cornhole Boards. Category: Outdoor Games.
..                                                                                                 ...
162                                   Brand: Nike. Gender: Men's. Subcategory: Shirts. Category: Golf.
163                         Brand: Nike. Gender: Unisex. Subcategory: Plates. Category: Weightlifting.
164                                 Brand: Nike. Gender: Men's. Subcatego

In [56]:
model_id =  "sentence-transformers__all-minilm-l6-v2"
# ingest pipeline definition
PIPELINE_ID="vector_search"
#CLOUD_ID="dsg-search-dev-east:ZWFzdHVzLmF6dXJlLmVsYXN0aWMtY2xvdWQuY29tOjQ0MyQ1Zjc1NDk5MTVmNmE0ZTBjYWFhYzYzM2IwZDI1MjNlZSRiZTg1OWU2ZGM0ZTg0Y2RmYTVjNDNkNzJiZDA3ZTg3Mg=="
HUB_MODEL_ID="intfloat/e5-small"
ES_HOST = os.getenv("ELASTIC_HOST")
ES_USER = os.getenv("ELASTIC_USERNAME")
ES_PASS = os.getenv("ELASTIC_PASSWORD")
search_type='vector'
model_name="sbert"
# define index name
INDEX_NAME = "pp_vs_embeddings"

# flag to check if index has to be deleted before creating
SHOULD_DELETE_INDEX = True

## Connect to Elasticsearch cluster

Create a elasticsearch client instance with your deployment `Cloud Id` and `API Key`. In this example, we are using the `API_KEY` and `CLOUD_ID` value from previous step.

Alternately you could use your deployment `Username` and `Password` to authenticate your instance.

In [57]:
es = Elasticsearch(
    ES_HOST,  # Replace with your actual host
    basic_auth=(ES_USER,ES_PASS),
    request_timeout=600,
    verify_certs=True  # Optional: Set to False only if using self-signed certs
)

# Test connection
print(es.info().body)

{'name': 'instance-0000000097', 'cluster_name': '5f7549915f6a4e0caaac633b0d2523ee', 'cluster_uuid': 'hrds7YuPRLy60DbQMfN9Ow', 'version': {'number': '8.17.6', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'dbcbbbd0bc4924cfeb28929dc05d82d662c527b7', 'build_date': '2025-04-30T14:07:12.231372970Z', 'build_snapshot': False, 'lucene_version': '9.12.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


## Deploy an NLP model to ES (ONE TIME ONLY)

We are using the [`eland`](https://www.elastic.co/guide/en/elasticsearch/client/eland/current/overview.html) tool to install a `text_embedding` model. For our model, We have used [`all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) to transform the search text into the dense vector.

The model will transfer your search query into vector which will be used for the search over the set of documents stored in Elasticsearch.


## Install text embedding NLP model

Using the [`eland_import_hub_model`](https://www.elastic.co/guide/en/elasticsearch/client/eland/current/machine-learning.html#ml-nlp-pytorch) script,  download and install `all-MiniLM-L6-v2` transformer model. Setting the NLP `--task-type` as `text_embedding`.

To get the cloud id, go to [Elastic cloud](https://cloud.elastic.co) and `On the deployment overview page, copy down the Cloud ID.`

To authenticate your request, You could use [API key](https://www.elastic.co/guide/en/kibana/current/api-keys.html#create-api-key). Alternatively, you can use your cloud deployment username and password.

More documentation: [machine learning](https://www.elastic.co/docs/reference/elasticsearch/clients/eland/machine-learning#ml-nlp-pytorch-auth)

In [58]:
# Check if it's already deployed
try:
    es.ml.start_trained_model_deployment(model_id=model_id)
    print(f"✅ Model '{model_id}' is deployed.")
except Exception as e:
    print(f"⚠️ Deployment may already exist or failed: {e}")

⚠️ Deployment may already exist or failed: BadRequestError(400, 'status_exception', 'Could not start model deployment because an existing deployment with the same id [sentence-transformers__all-minilm-l6-v2] exist')


DO NOT RUN BELOW STEP IF IT IS ALREADY DEPLOYED

In [59]:
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#finding-your-cloud-id
#ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")

# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key
#ELASTIC_API_KEY = getpass("Elastic Api Key: ")

In [60]:
# Load a Hugging Face transformers model directly from the model hub
#tm = TransformerModel( model_id = "intfloat/e5-small", task_type = "text_embedding")


# # Export the model in a TorchScrpt representation which Elasticsearch uses
#tmp_path = "my_vs_models"
#full_path = os.path.abspath(tmp_path)
#Path(tmp_path).mkdir(parents=True, exist_ok=True)
#print(f"Model will be saved to: {full_path}")
#model_path, config, vocab_path = tm.save(tmp_path)

In [61]:
#ptm = PyTorchModel(es, tm.elasticsearch_model_id())
#ptm.import_model(model_path=model_path, config_path=None, vocab_path=vocab_path, config=config)


Below is the right API to deploy any model. Now it is throwing Security Exception Error. Uncomment this when the issue is resolved.
elasticsearch.AuthenticationException: AuthenticationException(401, 'security_exception', 'unable to authenticate user [data_ingestion] for REST request [/]')

In [62]:
#!eland_import_hub_model \
#    --hub-model-id HUB_MODEL_ID \
#    --task-type text_embedding \
#    --cloud-id CLOUD_ID \
#    -u ES_USER \
#    -p ES_PASS \
#    --es-model-id model_id

## Create an Ingest pipeline

We need to create a text embedding ingest pipeline to generate vector (text) embeddings for `item + attribute name value pairs` field.

The pipeline below is defining a processor for the [inference](https://www.elastic.co/guide/en/elasticsearch/reference/current/inference-processor.html) to the NLP model.

In [63]:
def get_index_mapping(search_type: str = search_type, model_name: str = "sbert") -> dict:
    """Generate index mapping for vector, sparse, or hybrid search."""

    # Common text fields used in all modes except pure vector
    text_fields = {
        "title": {
            "type": "text",
            "fields": {
                "keyword": {"type": "keyword", "ignore_above": 256}
            }
        },
        "flat_description": {
            "type": "text",
            "fields": {
                "keyword": {"type": "keyword", "ignore_above": 256}
            }
        },
        "embedding_attributes": {
            "type": "text",
            "fields": {
                "keyword": {"type": "keyword", "ignore_above": 256}
            }
        },
    }

    # Determine vector field name based on model
    if model_name.lower() == "sbert":
        vector_field_name = "vs_sbert_embedding"
    else:
        vector_field_name = "vs_e5_embedding"

    # Dense vector field
    vector_field = {
        vector_field_name: {
            "properties": {
                "is_truncated": {"type": "boolean"},
                "model_id": {
                    "type": "text",
                    "fields": {"keyword": {"type": "keyword", "ignore_above": 256}},
                },
                "predicted_value": {
                    "type": "dense_vector",
                    "dims": 384,
                    "index": True,
                    "similarity": "cosine",
                },
            }
        }
    }

    # Final mapping logic
    if search_type == "vector":
        properties = vector_field
    elif search_type == "sparse":
        properties = text_fields
    else:  # hybrid
        properties = {**text_fields, **vector_field}

    return {
        "properties": properties
    }


In [64]:
# Hybrid (default)
#INDEX_MAPPING = get_index_mapping()  

# Vector-only
INDEX_MAPPING = get_index_mapping(search_type=search_type, model_name="sbert")

# Sparse/text-only
#INDEX_MAPPING = get_index_mapping(search_type="sparse")


In [65]:
# Function to get embedding from ES
def get_embedding(text: str, es_client: Elasticsearch, model_id: str) -> list[float]:
    try:
        response = es_client.ml.infer_trained_model(
            model_id=model_id,
            body={"docs": [{"text_field": text}]}
        )
        return response['inference_results'][0]['predicted_value']
    except Exception as e:
        print(f"[Error] Text: '{text[:30]}...' => {e}")
        return [0.0] * 384  # Return zero-vector if inference fails


#sample embedding
#embedding = model.encode("Item: Nike Running Shoes. Brand: Nike. Color: Black. Size: 10. Category: Shoes.")

In [66]:
# Apply to DataFrame
descriptions = df['flat_description'].fillna("")
embeddings = []

for desc in tqdm(descriptions, desc="Embedding via ES"):
    embedding = get_embedding(desc, es, model_id)
    embeddings.append(embedding)

df['vs_sbert_embedding'] = embeddings

Embedding via ES: 100%|██████████████████████████████████████████████████████████████████████████████████████| 166/166 [00:16<00:00, 10.09it/s]


In [67]:
print(df[['flat_description', 'vs_sbert_embedding']].head(1))
print(len(df['vs_sbert_embedding'].iloc[0]))

                                    flat_description  \
0  Brand: Rocky Mountain Tackle. Gender: Unisex. ...   

                                  vs_sbert_embedding  
0  [-0.058064743876457214, 0.010954274795949459, ...  
384


## Create Index with mappings

We will now create an elasticsearch index with correct mapping before we index documents.
We are adding `vs_sb_embedding` to include the `model_id` and `predicted_value` to store the embeddings.

✅ dense_vector with index: true and similarity: cosine is the right setup for native ES8 ANN search.

In [68]:
INDEX_SETTINGS = {
    "index": {
        "number_of_replicas": "0",
        "number_of_shards": "1",
        #"knn": True  # Enable kNN search
        # No need for "knn": true — only needed if using legacy knn_vector
    }
}

# check if we want to delete index before creating the index
if SHOULD_DELETE_INDEX:
    if es.indices.exists(index=INDEX_NAME):
        print("Deleting existing %s" % INDEX_NAME)
        es.indices.delete(index=INDEX_NAME, ignore=[400, 404])

print("Creating index %s" % INDEX_NAME)
es.indices.create(
    index=INDEX_NAME, mappings=INDEX_MAPPING, settings=INDEX_SETTINGS, ignore=[400, 404]
)

/var/folders/36/3pkr81w50c72tdrmbywpc2540000gp/T/ipykernel_36166/1054438030.py:14: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  es.indices.delete(index=INDEX_NAME, ignore=[400, 404])


Deleting existing pp_vs_embeddings
Creating index pp_vs_embeddings


/var/folders/36/3pkr81w50c72tdrmbywpc2540000gp/T/ipykernel_36166/1054438030.py:17: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  es.indices.create(


ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'pp_vs_embeddings'})

In [69]:
df.head()

,stringFacets,name,_id,flat_description,vs_sbert_embedding
0,"[{'identifier': 'X_BRAND', 'value': 'Rocky Mou...",Rocky Mountain Tackle Signature Dodger,21RMAURCKYMTNDDGRFAC,Brand: Rocky Mountain Tackle. Gender: Unisex. ...,"[-0.058064743876457214, 0.010954274795949459, ..."
1,"[{'identifier': 'X_BRAND', 'value': 'Rocky Mou...",Rocky Mountain Growler 20 Microshift Bike,23NFKAGRWLR20MCRSPRF,Brand: Rocky Mountain. Gender: Men's. Subcateg...,"[-0.04425465315580368, 0.056473180651664734, -..."
2,"[{'identifier': 'X_BRAND', 'value': 'Rocky Mou...",Rocky Mountain Instinct Powerplay Alloy 30,23NFKUNSTNCTPPLLYBAC,Brand: Rocky Mountain. Gender: Men's. Subcateg...,"[-0.04425465315580368, 0.056473180651664734, -..."
3,"[{'identifier': '320', 'storeId': 10001, 'part...",Rocky Mountain Tackle Signature Dodger,13502535,Brand: Rocky Mountain Tackle. Gender: Unisex. ...,"[-0.058064743876457214, 0.010954274795949459, ..."
4,"[{'identifier': '3232', 'storeId': 10001, 'par...",Elakai Travel 1' x 2' Cornhole Boards with Car...,25353984,Brand: Elakai. Gender: Unisex. Subcategory: Co...,"[0.042894989252090454, 0.016177961602807045, -..."


✅ Key Requirements for Hybrid Search
To use both:

BM25 (text search) → Fields must be of type "text" in the mapping.

Vector search → Embedding must be stored under a dense_vector field with proper indexing.


## Index data to elasticsearch index

Let's index sample data using the ingest pipeline.

Note: Before we begin indexing, ensure you have [started your trained model deployment](https://www.elastic.co/guide/en/machine-learning/current/ml-nlp-deploy-model.html).

In [70]:

# For 'title' or any string columns:
df['name'] = df['name'].replace({np.nan: ''})

# Or more generally, for all object columns:
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].fillna('')


In [71]:
df.head()

,stringFacets,name,_id,flat_description,vs_sbert_embedding
0,"[{'identifier': 'X_BRAND', 'value': 'Rocky Mou...",Rocky Mountain Tackle Signature Dodger,21RMAURCKYMTNDDGRFAC,Brand: Rocky Mountain Tackle. Gender: Unisex. ...,"[-0.058064743876457214, 0.010954274795949459, ..."
1,"[{'identifier': 'X_BRAND', 'value': 'Rocky Mou...",Rocky Mountain Growler 20 Microshift Bike,23NFKAGRWLR20MCRSPRF,Brand: Rocky Mountain. Gender: Men's. Subcateg...,"[-0.04425465315580368, 0.056473180651664734, -..."
2,"[{'identifier': 'X_BRAND', 'value': 'Rocky Mou...",Rocky Mountain Instinct Powerplay Alloy 30,23NFKUNSTNCTPPLLYBAC,Brand: Rocky Mountain. Gender: Men's. Subcateg...,"[-0.04425465315580368, 0.056473180651664734, -..."
3,"[{'identifier': '320', 'storeId': 10001, 'part...",Rocky Mountain Tackle Signature Dodger,13502535,Brand: Rocky Mountain Tackle. Gender: Unisex. ...,"[-0.058064743876457214, 0.010954274795949459, ..."
4,"[{'identifier': '3232', 'storeId': 10001, 'par...",Elakai Travel 1' x 2' Cornhole Boards with Car...,25353984,Brand: Elakai. Gender: Unisex. Subcategory: Co...,"[0.042894989252090454, 0.016177961602807045, -..."


In [72]:
def generate_docs(df, search_type=search_type, model_id=None):
    for _, row in df.iterrows():
        doc = {}

        # Add sparse fields
        if search_type in ["sparse", "hybrid"]:
            doc["title"] = row["name"]
            doc["flat_description"] = row["flat_description"]
            doc["embedding_attributes"] = row.get("embedding_attributes", "")

        # Add vector field
        if search_type in ["vector", "hybrid"]:
            doc["title"] = row["name"]
            doc["vs_sbert_embedding"] = {
                "predicted_value": row["vs_sbert_embedding"],  # 384-dim list
                "model_id": model_id,
                "is_truncated": False
            }

        yield {
            "_index": INDEX_NAME,
            "_id": row["_id"],
            "_source": doc
        }


In [73]:
try:
    helpers.bulk(es, generate_docs(df, search_type=search_type, model_id=model_id))
except BulkIndexError as e:
    for error in e.errors[:5]:  # Show first 5 errors
        print(error)

## Querying the dataset
The next step is to run a query to search for relevant blogs. The example query searches for `"model_text": "how to track network connections"` using the model we uploaded to Elasticsearch `sentence-transformers__all-minilm-l6-v2`.

The process is one query even though it internally consists of two tasks. First, the query will generate an vector for your search text using the NLP model and then pass this vector to search over the dataset.

As a result, the output shows the list of query documents sorted by their proximity to the search query.

🔧 Notes  
The + 1.0 in vector similarity is to ensure scores are positive (Elasticsearch requires non-negative scores for script_score).

In sparse-only mode, query_vector isn’t needed at all.

In vector-only mode, you can also drop _score entirely unless you need it for a combination.


In [74]:
query_text = "lightweight nike running shoes for men"

response = es.ml.infer_trained_model(
    model_id=model_id,
    body={"docs": [{"text_field": query_text}]}
)

query_vector = response.body["inference_results"][0]["predicted_value"]


In [75]:
def build_query_body(query_text: str, query_vector: list[float], search_type: str = search_type, size: int = 5):
    if search_type == "vector":
        return {
            "size": size,
            "query": {
                "script_score": {
                    "query": {"match_all": {}},
                    "script": {
                        "source": "cosineSimilarity(params.query_vector, 'vs_sbert_embedding.predicted_value') + 1.0",
                        "params": {
                            "query_vector": query_vector
                        }
                    }
                }
            }
        }

    elif search_type == "sparse":
        return {
            "size": size,
            "query": {
                "multi_match": {
                    "query": query_text,
                    "fields": ["_id", "title", "flat_description"]
                }
            }
        }

    else:  # hybrid
        return {
            "size": size,
            "query": {
                "script_score": {
                    "query": {
                        "multi_match": {
                            "query": query_text,
                            "fields": ["_id", "title", "flat_description"]
                        }
                    },
                    "script": {
                        "source": """
                            double cosineSim = cosineSimilarity(params.query_vector, 'vs_sbert_embedding.predicted_value');
                            return _score * 0.5 + (cosineSim + 1.0) * 0.5;
                        """,
                        "params": {
                            "query_vector": query_vector
                        }
                    }
                }
            }
        }


# Example Elasticsearch query vector search
Note: Adding +1.0 avoids negative scores from cosine similarity.

In [76]:
query_body = build_query_body(query_text, query_vector, search_type=search_type)

In [77]:
# Determine which embedding field to remove
embedding_field = "vs_sbert_embedding" if model_name == "sbert" else "vs_e5_embedding"

response = es.search(index=INDEX_NAME, body=query_body)

results = []
for hit in response['hits']['hits']:
    source = hit['_source'].copy()
    source.pop('vs_sbert_embedding', None)  # Remove embedding if present
    
    result = {
        "score": hit['_score'],
        "title": source.get('title', 'N/A'),
        "id": hit.get('_id', 'N/A'),  
        "attributes": source
    }
    results.append(result)

print(json.dumps(results, indent=2))


[]


# Add filters Example  

keyword enables exact matching on a text field.

vs_sb_embedding.model_id.keyword = "all-MiniLM-L6-v2" filters for documents using that specific embedding model.

It’s useful for multi-model environments or ensuring vector-query consistency. Also filtering by attribute values like brand.

Below code enables us to query hugging face senetnce bert or E5 based on model id.

In [78]:
def build_query_body(query_text: str, query_vector: list[float], search_type: str = search_type, size: int = 5):
    filters = []

    # Filter only applies when vector field exists
    if search_type in ["vector", "hybrid"]:
        filters.append({"term": {"vs_sbert_embedding.model_id.keyword": model_id}})

    # ----------------------
    if search_type == "vector":
        return {
            "size": size,
            "query": {
                "script_score": {
                    "query": {
                        "bool": {
                            "filter": filters or [{"match_all": {}}]
                        }
                    },
                    "script": {
                        "source": "cosineSimilarity(params.query_vector, 'vs_sbert_embedding.predicted_value') + 1.0",
                        "params": {
                            "query_vector": query_vector
                        }
                    }
                }
            }
        }

    elif search_type == "sparse":
        return {
            "size": size,
            "query": {
                "bool": {
                    "must": [
                        {
                            "multi_match": {
                                "query": query_text,
                                "fields": ["title", "flat_description"]
                            }
                        }
                    ],
                    "filter": filters
                }
            }
        }

    else:  # hybrid
        return {
            "size": size,
            "query": {
                "script_score": {
                    "query": {
                        "bool": {
                            "must": [
                                {
                                    "multi_match": {
                                        "query": query_text,
                                        "fields": ["title", "flat_description"]
                                    }
                                }
                            ],
                            "filter": filters
                        }
                    },
                    "script": {
                        "source": """
                            double cosineSim = cosineSimilarity(params.query_vector, 'vs_sbert_embedding.predicted_value');
                            return _score * 0.5 + (cosineSim + 1.0) * 0.5;
                        """,
                        "params": {
                            "query_vector": query_vector
                        }
                    }
                }
            }
        }


In [79]:
query_body = build_query_body(query_text, query_vector, search_type=search_type)

response = es.search(index=INDEX_NAME, body=query_body)
print(f"Total hits: {response['hits']['total']['value']}")
for hit in response["hits"]["hits"]:
    print(f"Score: {hit['_score']:.4f} | Title: {hit['_source'].get('title', 'N/A')}")


Total hits: 0


In [80]:

response = es.search(index=INDEX_NAME, body=query_body)

results = []
for hit in response['hits']['hits']:
    source = hit['_source'].copy()
    source.pop('vs_sbert_embedding', None)  # Remove embedding if present
    
    result = {
        "score": hit['_score'],
        "title": source.get('title', 'N/A'),
        "attributes": source
    }
    results.append(result)

print(json.dumps(results, indent=2))


[]


# NEXT Model: E5. 
**As per Elastic Search Consultant, did not add "passage" to any text**

In [81]:
model_id="intfloat__e5-small" #change the model to E5
SHOULD_DELETE_INDEX=False
# Vector-only
INDEX_MAPPING = get_index_mapping(search_type=search_type, model_name="e5")

# Apply to DataFrame
descriptions = df['flat_description'].fillna("")
embeddings = []

for desc in tqdm(descriptions, desc="Embedding via ES"):
    embedding = get_embedding(desc, es, model_id)
    embeddings.append(embedding)

df['vs_e5_embedding'] = embeddings

Embedding via ES: 100%|██████████████████████████████████████████████████████████████████████████████████████| 166/166 [00:16<00:00,  9.99it/s]


In [82]:
def generate_docs(df):
    for _, row in df.iterrows():
        yield {
            "_index": INDEX_NAME,
            "_id": row["_id"],
            "_source": {
                "title": row["name"],
                "flat_description": row["flat_description"],
                "embedding_attributes": row.get("embedding_attributes", ""),
                "vs_e5_embedding": {
                    "is_truncated": True,
                    "model_id": model_id,
                    "predicted_value": row["vs_e5_embedding"],  
                }
            }
        }


In [83]:
# check if we want to delete index before creating the index
if SHOULD_DELETE_INDEX:
    if es.indices.exists(index=INDEX_NAME):
        print("Deleting existing %s" % INDEX_NAME)
        es.indices.delete(index=INDEX_NAME, ignore=[400, 404])

print("Creating index %s" % INDEX_NAME)
es.indices.create(
    index=INDEX_NAME, mappings=INDEX_MAPPING, settings=INDEX_SETTINGS, ignore=[400, 404]
)

# Bulk index
try:
    helpers.bulk(es, generate_docs(df))
    print("indexing completed")
except BulkIndexError as e:
    for error in e.errors[:5]:  # Show first 5 errors
        print(error)

/var/folders/36/3pkr81w50c72tdrmbywpc2540000gp/T/ipykernel_36166/2143641811.py:8: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  es.indices.create(


Creating index pp_vs_embeddings
indexing completed


In [84]:
df.head()

,stringFacets,name,_id,flat_description,vs_sbert_embedding,vs_e5_embedding
0,"[{'identifier': 'X_BRAND', 'value': 'Rocky Mou...",Rocky Mountain Tackle Signature Dodger,21RMAURCKYMTNDDGRFAC,Brand: Rocky Mountain Tackle. Gender: Unisex. ...,"[-0.058064743876457214, 0.010954274795949459, ...","[0.02724776417016983, -0.04879041761159897, -0..."
1,"[{'identifier': 'X_BRAND', 'value': 'Rocky Mou...",Rocky Mountain Growler 20 Microshift Bike,23NFKAGRWLR20MCRSPRF,Brand: Rocky Mountain. Gender: Men's. Subcateg...,"[-0.04425465315580368, 0.056473180651664734, -...","[-0.004020625725388527, -0.022048858925700188,..."
2,"[{'identifier': 'X_BRAND', 'value': 'Rocky Mou...",Rocky Mountain Instinct Powerplay Alloy 30,23NFKUNSTNCTPPLLYBAC,Brand: Rocky Mountain. Gender: Men's. Subcateg...,"[-0.04425465315580368, 0.056473180651664734, -...","[-0.004020625725388527, -0.022048858925700188,..."
3,"[{'identifier': '320', 'storeId': 10001, 'part...",Rocky Mountain Tackle Signature Dodger,13502535,Brand: Rocky Mountain Tackle. Gender: Unisex. ...,"[-0.058064743876457214, 0.010954274795949459, ...","[0.02724776417016983, -0.04879041761159897, -0..."
4,"[{'identifier': '3232', 'storeId': 10001, 'par...",Elakai Travel 1' x 2' Cornhole Boards with Car...,25353984,Brand: Elakai. Gender: Unisex. Subcategory: Co...,"[0.042894989252090454, 0.016177961602807045, -...","[-0.014461016282439232, -0.034393440932035446,..."


In [86]:
# Step 1: Generate embedding for the query
query_text = "lightweight nike running shoes for men"

inference_response = es.ml.infer_trained_model(
    model_id=model_id,
    body={"docs": [{"text_field": query_text}]}
)

query_vector = inference_response["inference_results"][0]["predicted_value"]

# Step 2: Build pure vector search query
query_body = {
    "size": 5,
    "query": {
        "script_score": {
            "query": {
                "match_all": {}   #only vector search query
            },
            "script": {
                "source": "cosineSimilarity(params.query_vector, 'vs_e5_embedding.predicted_value') + 1.0",
                "params": {
                    "query_vector": query_vector
                }
            }
        }
    }
}

# Step 3: Run the vector search
search_response = es.search(index=INDEX_NAME, body=query_body)

# Step 4: Parse and print results
for hit in search_response["hits"]["hits"]:
    print(f"Score: {hit['_score']:.4f} | Title: {hit['_source']['title']}")


Score: 1.8756 | Title: Nike Men's Str8Jacket Football Cleat Spat System
Score: 1.8756 | Title: Nike+ Sportwatch GPS Powered by TomTom with Sensor
Score: 1.8699 | Title: Nike Kids' Toddler Flex Advance Running Shoes
Score: 1.8656 | Title: 
Score: 1.8558 | Title: Nike Show X2 Sunglasses


In [87]:
response = es.search(index=INDEX_NAME, body=query_body)

results = []
for hit in response['hits']['hits']:
    source = hit['_source'].copy()
    source.pop('vs_e5_embedding', None)  # Remove embedding if present
    
    result = {
        "score": hit['_score'],
        "title": source.get('title', 'N/A'),
        "id":hit["_id"]
    }
    results.append(result)

print(json.dumps(results, indent=2))


[
  {
    "score": 1.8756247,
    "title": "Nike Men's Str8Jacket Football Cleat Spat System",
    "id": "16NIKMSTR8JCKTSPTGSA"
  },
  {
    "score": 1.8756247,
    "title": "Nike+ Sportwatch GPS Powered by TomTom with Sensor",
    "id": "16NIKUNKSPRTWTCHGTCH"
  },
  {
    "score": 1.8699386,
    "title": "Nike Kids' Toddler Flex Advance Running Shoes",
    "id": "21NIKYFLXDVNCHLYLBYS"
  },
  {
    "score": 1.8656139,
    "title": "",
    "id": "17NIKWTMPTRCK35XXAPB"
  },
  {
    "score": 1.8557814,
    "title": "Nike Show X2 Sunglasses",
    "id": "16NIKUSHWX2MTTBSDSGS"
  }
]


### Future considerations (Not implemented)
1. It seems like it could be a good idea to compare the performance for the actual description embedding vs the custom description embedding. I don't think we currently have that as part of the plan but it might be an interesting exploration sometime in the future.

2. Parallelize the calls (for large datasets)

3. Cache results for repeated inputs

4. Add logging or metrics